<a href="https://colab.research.google.com/github/Projit1101/Financial-fraud-classification/blob/main/models/graphsage_radius_optimized.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Check if GPU is available

In [ ]:
import torch
print(torch.cuda.is_available())

True


Define the device (Output should be: cuda)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


Run this once per session (Access to my Google Drive)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Cell 1: Install Dependencies
# We can rely on Colab's pre-installed PyTorch and just install torch-geometric directly.
!pip install torch-geometric scikit-learn matplotlib pandas
!pip install faiss-cpu
!pip install pyg_lib torch_sparse torch_scatter -f https://data.pyg.org/whl/torch-$(python -c "import torch; print(torch.__version__)").html
!pip install optuna

In [ ]:
# Cell 2: Data Loading and Preprocessing
import pandas as pd
import torch
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Datasets/creditcard.csv')

print("Original Dataset Shape:", df.shape)
print("Fraudulent transactions:", len(df[df['Class'] == 1]))
print("Legitimate transactions:", len(df[df['Class'] == 0]))

# Separate features and labels
# We drop 'Time', 'Class' as they aren't predictive features in the same way
features = df.drop(columns=['Time', 'Class']).values
labels = df['Class'].values

# Standardize features (Mean = 0, Variance = 1)
scaler = StandardScaler()
features = scaler.fit_transform(features)

# Convert to PyTorch tensors
X = torch.tensor(features, dtype=torch.float32).to(device)
y = torch.tensor(labels, dtype=torch.long).to(device)

print("Feature Tensor Shape:", X.shape)
print("Label Tensor Shape:", y.shape)

Original Dataset Shape: (284807, 31)
Fraudulent transactions: 492
Legitimate transactions: 284315
Feature Tensor Shape: torch.Size([284807, 29])
Label Tensor Shape: torch.Size([284807])


~ 4 minutes

In [ ]:
# Cell 3: Capped Radius-Neighborhood Graph Construction
import faiss
import torch
import numpy as np
from torch_geometric.data import Data
from sklearn.model_selection import train_test_split

print("Creating Train/Val/Test splits...")
train_idx, temp_idx = train_test_split(
    np.arange(features.shape[0]), test_size=0.3, random_state=42, stratify=labels
)
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.5, random_state=42, stratify=labels[temp_idx]
)

def create_capped_radius_edge_index(features_np, train_indices, max_neighbors=100, radius=3.0):
    print(f"Building Capped Radius graph (Max {max_neighbors} neighbors, Radius {radius}) using FAISS...")
    d = features_np.shape[1]

    # 1. Build FAISS index on TRAINING data
    train_features = np.ascontiguousarray(features_np[train_indices], dtype=np.float32)
    index = faiss.IndexFlatL2(d)
    index.add(train_features)

    # 2. Safely find at most `max_neighbors` for all nodes (Prevents RAM crashes)
    all_features = np.ascontiguousarray(features_np, dtype=np.float32)
    distances, local_neighbor_indices = index.search(all_features, max_neighbors)

    # FAISS returns squared L2 distances
    radius_squared = radius ** 2

    # 3. Create boolean mask for valid neighbors (within radius and not placeholder -1)
    valid_mask = (distances <= radius_squared) & (local_neighbor_indices != -1)

    # 4. Extract valid rows (receivers) and cols (senders)
    rows = np.repeat(np.arange(features_np.shape[0]), max_neighbors)
    cols = train_indices[local_neighbor_indices.flatten()]

    # Apply the mask to filter out distant neighbors
    flat_mask = valid_mask.flatten()
    rows = rows[flat_mask]
    cols = cols[flat_mask]

    # Filter out self-loops
    self_loop_mask = rows != cols
    rows = rows[self_loop_mask]
    cols = cols[self_loop_mask]

    # Direction: Train (cols) -> All Nodes (rows)
    edge_index = torch.tensor(np.vstack((cols, rows)), dtype=torch.long).to(device)
    return edge_index

# We use a smaller radius and cap the max neighbors to 100 to protect RAM
edge_index = create_capped_radius_edge_index(features, train_idx, max_neighbors=100, radius=3.0)

train_mask = torch.zeros(features.shape[0], dtype=torch.bool).to(device)
train_mask[train_idx] = True
val_mask = torch.zeros(features.shape[0], dtype=torch.bool).to(device)
val_mask[val_idx] = True
test_mask = torch.zeros(features.shape[0], dtype=torch.bool).to(device)
test_mask[test_idx] = True

data = Data(x=X, edge_index=edge_index, y=y,
            train_mask=train_mask, val_mask=val_mask, test_mask=test_mask)

print("\nGraph Conversion Successful!")
print(f"Number of Edges: {data.num_edges}")

Creating Train/Val/Test splits...
Building Capped Radius graph (Max 100 neighbors, Radius 3.0) using FAISS...

Graph Conversion Successful!
Number of Edges: 22256498


In [ ]:
# Cell 4: Define GraphSAGE Model and Loss Function
import torch
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
import torch.nn as nn

class GraphSAGE_FraudModel(nn.Module):
    def __init__(self, in_features, hidden_dim, out_features, dropout_rate=0.5):
        super(GraphSAGE_FraudModel, self).__init__()
        self.conv1 = SAGEConv(in_features, hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, out_features)
        self.dropout_rate = dropout_rate

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = self.conv1(x, edge_index)
        x = F.elu(x)
        x = F.dropout(x, p=self.dropout_rate, training=self.training)
        x = self.conv2(x, edge_index)
        return F.log_softmax(x, dim=1)

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt)**self.gamma * ce_loss
        return focal_loss.mean()

num_features = X.shape[1]
num_classes = 2

In [ ]:
# Cell 5: Initialize NeighborLoaders
from torch_geometric.loader import NeighborLoader

print("Creating NeighborLoaders...")

# Sample 15 neighbors in the first layer, 10 in the second
train_loader = NeighborLoader(
    data,
    num_neighbors=[15, 10],
    batch_size=2048,
    input_nodes=data.train_mask,
    shuffle=True
)

val_loader = NeighborLoader(
    data,
    num_neighbors=[15, 10],
    batch_size=2048,
    input_nodes=data.val_mask,
)

test_loader = NeighborLoader(
    data,
    num_neighbors=[15, 10],
    batch_size=2048,
    input_nodes=data.test_mask,
)

print("NeighborLoaders successfully created! You can now run the Optuna cell.")

Creating NeighborLoaders...
NeighborLoaders successfully created! You can now run the Optuna cell.


In [ ]:
# Cell 6: Hyperparameter Optimization with Optuna
import optuna
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import precision_recall_curve

def objective(trial):
    # 1. Suggest hyperparameters
    hidden_dim = trial.suggest_categorical('hidden_dim', [16, 32, 64])
    dropout_rate = trial.suggest_float('dropout_rate', 0.2, 0.6)
    lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    focal_alpha = trial.suggest_float('focal_alpha', 0.5, 0.9)
    focal_gamma = trial.suggest_float('focal_gamma', 1.0, 3.0)

    # 2. Initialize Model & Optimizer
    model = GraphSAGE_FraudModel(
        in_features=num_features,
        hidden_dim=hidden_dim,
        out_features=num_classes,
        dropout_rate=dropout_rate
    ).to(device)

    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
    criterion = FocalLoss(alpha=focal_alpha, gamma=focal_gamma)

    # 3. Train for fewer epochs to speed up the search
    epochs = 10  # Reduced for speed
    for epoch in range(epochs):
        model.train()
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            out = model(batch)
            loss = criterion(out[:batch.batch_size], batch.y[:batch.batch_size])
            loss.backward()
            optimizer.step()

    # 4. Evaluate on Validation Set
    model.eval()
    all_probs = []
    all_true = []
    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(device)
            out = model(batch)
            probs = torch.exp(out[:batch.batch_size, 1])
            all_probs.append(probs.cpu())
            all_true.append(batch.y[:batch.batch_size].cpu())

    y_prob_val = torch.cat(all_probs).numpy()
    y_true_val = torch.cat(all_true).numpy()

    # 5. Calculate Max F1 on Validation
    precisions, recalls, thresholds = precision_recall_curve(y_true_val, y_prob_val)
    f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
    max_f1 = np.max(f1_scores)

    return max_f1

# Run the optimization study
print("Starting Optuna Hyperparameter Search...")
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=15)

print("\nBest Trial:")
print(f"  Validation F1: {study.best_trial.value:.4f}")
print("  Params: ")
for key, value in study.best_trial.params.items():
    print(f"    {key}: {value}")

# Save the best parameters to global variables for the next cell
best_params = study.best_trial.params
best_hidden_dim = best_params['hidden_dim']
best_dropout_rate = best_params['dropout_rate']
best_lr = best_params['lr']
best_alpha = best_params['focal_alpha']
best_gamma = best_params['focal_gamma']

print("\nBest parameters saved to memory for final training.")

[I 2026-04-01 15:03:53,682] A new study created in memory with name: no-name-b4a9859f-3e12-435f-9c1a-dc41a3d7d88e


Starting Optuna Hyperparameter Search...


[I 2026-04-01 15:05:17,197] Trial 0 finished with value: 0.7538461489420119 and parameters: {'hidden_dim': 64, 'dropout_rate': 0.5200616192819403, 'lr': 0.001063794257921641, 'focal_alpha': 0.6495016079012077, 'focal_gamma': 2.0903889073454396}. Best is trial 0 with value: 0.7538461489420119.
[I 2026-04-01 15:06:36,758] Trial 1 finished with value: 0.7785234849331112 and parameters: {'hidden_dim': 64, 'dropout_rate': 0.5261621697193315, 'lr': 0.0024118069555048023, 'focal_alpha': 0.7321592774160257, 'focal_gamma': 2.8534187813239353}. Best is trial 1 with value: 0.7785234849331112.
[I 2026-04-01 15:07:51,321] Trial 2 finished with value: 0.6751592306852205 and parameters: {'hidden_dim': 16, 'dropout_rate': 0.5080383219439315, 'lr': 0.0001564495349511693, 'focal_alpha': 0.6271963865297061, 'focal_gamma': 2.3468601411239343}. Best is trial 1 with value: 0.7785234849331112.
[I 2026-04-01 15:09:05,841] Trial 3 finished with value: 0.5999999950281251 and parameters: {'hidden_dim': 16, 'drop


Best Trial:
  Validation F1: 0.7785
  Params: 
    hidden_dim: 64
    dropout_rate: 0.5261621697193315
    lr: 0.0024118069555048023
    focal_alpha: 0.7321592774160257
    focal_gamma: 2.8534187813239353

Best parameters saved to memory for final training.


## Model Training and Evaluation Results

In [13]:
# Cell 7: Train Final Model and Evaluate
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, precision_recall_curve

# --- 1. Train Final Model with Best Parameters ---
# The variables below are automatically pulled from Cell 5
final_model = GraphSAGE_FraudModel(
    in_features=num_features,
    hidden_dim=best_hidden_dim,
    out_features=num_classes,
    dropout_rate=best_dropout_rate
).to(device)

optimizer = optim.Adam(final_model.parameters(), lr=best_lr, weight_decay=5e-4)
criterion = FocalLoss(alpha=best_alpha, gamma=best_gamma)

print("Training final model with best hyperparameters...")
epochs = 50 # Train for full epochs now
for epoch in range(epochs):
    final_model.train()
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = final_model(batch)
        loss = criterion(out[:batch.batch_size], batch.y[:batch.batch_size])
        loss.backward()
        optimizer.step()

# --- 2. Evaluate logic ---
def get_loader_predictions(loader, model):
    model.eval()
    all_probs = []
    all_true = []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            out = model(batch)
            probs = torch.exp(out[:batch.batch_size, 1])
            all_probs.append(probs.cpu())
            all_true.append(batch.y[:batch.batch_size].cpu())
    return torch.cat(all_probs).numpy(), torch.cat(all_true).numpy()

# Tune Threshold on Validation
y_prob_val, y_true_val = get_loader_predictions(val_loader, final_model)
precisions, recalls, thresholds = precision_recall_curve(y_true_val, y_prob_val)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
best_idx = np.argmax(f1_scores)
optimal_threshold = thresholds[best_idx]

print(f"\n--- Tuning on Validation Set ---")
print(f"Optimal threshold for max F1 is: {optimal_threshold:.4f}")

# Final Evaluation on Test Set
y_prob_test, y_true_test = get_loader_predictions(test_loader, final_model)
y_pred_test = (y_prob_test >= optimal_threshold).astype(int)

print(f"\n--- Final Unbiased Test Set Results ---")
print(f"Accuracy:  {accuracy_score(y_true_test, y_pred_test):.4f}")
print(f"Precision: {precision_score(y_true_test, y_pred_test, zero_division=1):.4f}")
print(f"Recall:    {recall_score(y_true_test, y_pred_test, zero_division=1):.4f}")
print(f"F1 Score:  {f1_score(y_true_test, y_pred_test, zero_division=1):.4f}")
print(f"AUC-ROC:   {roc_auc_score(y_true_test, y_prob_test):.4f}")

Training final model with best hyperparameters...

--- Tuning on Validation Set ---
Optimal threshold for max F1 is: 0.4554

--- Final Unbiased Test Set Results ---
Accuracy:  0.9993
Precision: 0.8028
Recall:    0.7703
F1 Score:  0.7862
AUC-ROC:   0.9620


In [14]:
# Cell 7: Save and Download the Optimized Model
import torch
from google.colab import files

model_save_path = 'graphSAGE_radius_optimized.pth'
torch.save(final_model.state_dict(), model_save_path)
print(f"Model saved as: {model_save_path}")

files.download(model_save_path)

Model saved as: graphSAGE_radius_optimized.pth


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>